# TP10 — Analyse exploratoire de référence

**Objectif** : comprendre ce que contient le périmètre NutriScope avant de construire les modèles.
Chaque graphique répond à une question, affirme un message dans son titre, et est suivi de 3 lignes d'interprétation.

**Données** : `data/food_perimeter.parquet` nettoyé par `src/cleaning.prepare_data` (456 479 produits vendus en France, 9 rayons).
Source : Open Food Facts, licence ODbL.

| Partie | Question |
|---|---|
| 1 | Que mange-t-on réellement dans le périmètre ? |
| 2 | Où les données sont-elles fiables ? |
| 3 | Quels nutriments vont avec le Nutri-Score ? |
| 4 | Valeurs extrêmes : vraies ou erreurs ? |

In [ ]:
import sys
from pathlib import Path

# Racine du projet : le dossier parent de notebooks/
PROJECT_ROOT = Path.cwd().parent.parent
SAVING_PATH = PROJECT_ROOT / "data" / "outputs"
SAVING_PATH.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(PROJECT_ROOT))

import matplotlib.pyplot as plt
import pandas as pd

from src.cleaning import prepare_data

# Couleurs : une seule teinte pour les données, des gris discrets pour le reste
SURFACE = "#fcfcfb"
INK = "#0b0b0b"
INK_SECONDARY = "#52514e"
MUTED = "#898781"
GRID = "#e1e0d9"
BASELINE = "#c3c2b7"
BLUE = "#2a78d6"
BLUE_LIGHT = "#cde2fb"   # intérieur des boîtes
BLUE_DARK = "#184f95"    # trait de la médiane

plt.rcParams.update({
    "figure.facecolor": SURFACE,
    "axes.facecolor": SURFACE,
    "axes.edgecolor": BASELINE,
    "axes.labelcolor": INK_SECONDARY,
    "axes.titlecolor": INK,
    "xtick.color": MUTED,
    "ytick.color": INK_SECONDARY,
    "axes.grid": True,
    "grid.color": GRID,
    "grid.linewidth": 0.8,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.family": "Segoe UI",
    "font.size": 10,
})

# Noms des rayons à l'affichage (la base garde "en:...", comme décidé au TP4)
FOOD_GROUP_LABELS = {
    "en:sugary-snacks": "Snacks sucrés",
    "en:salty-snacks": "Snacks salés",
    "en:beverages": "Boissons",
    "en:fruits-and-vegetables": "Fruits et légumes",
    "en:milk-and-dairy-products": "Produits laitiers",
    "en:cereals-and-potatoes": "Céréales et pommes de terre",
    "en:composite-foods": "Plats composés",
    "en:fats-and-sauces": "Matières grasses et sauces",
    "en:fish-meat-eggs": "Viande, poisson, œufs",
}

In [ ]:
df = prepare_data(pd.read_parquet("../../data/food_perimeter.parquet"))
df["rayon"] = df["food_group"].map(FOOD_GROUP_LABELS)
df.shape

In [ ]:
def fmt_number(value, decimals=0):
    """Nombre à la française : 456479 -> '456 479', 21.6 -> '21,6'."""
    text = f"{value:,.{decimals}f}"
    return text.replace(",", " ").replace(".", ",")


def add_titles(fig, title, subtitle):
    """Ajoute le titre-message, le sous-titre (comment lire) et la source à une figure."""
    fig.suptitle(title, x=0.01, ha="left", fontsize=14, fontweight="bold", color=INK)
    fig.text(0.01, 0.905, subtitle, ha="left", fontsize=9.5, color=INK_SECONDARY)
    fig.text(
        0.01, 0.015,
        "Source : Open Food Facts (ODbL), produits vendus en France, après nettoyage (src/cleaning.py).",
        ha="left", fontsize=8.5, color=MUTED,
    )

## Partie 1 — Que mange-t-on réellement dans le périmètre ?

In [ ]:
counts = df["rayon"].value_counts().sort_values()
shares = counts / counts.sum() * 100

fig, ax = plt.subplots(figsize=(10, 5.5))
ax.barh(shares.index, shares.values, color=BLUE, height=0.7)

for rayon, share in shares.items():
    label = f"{fmt_number(share, 1)} %  ({fmt_number(counts[rayon])} produits)"
    ax.text(share + 0.3, rayon, label, va="center", fontsize=9.5, color=INK_SECONDARY)

ax.set_xlim(0, shares.max() * 1.35)
ax.xaxis.set_visible(False)
ax.grid(False)
ax.spines["bottom"].set_visible(False)
ax.tick_params(axis="y", length=0)

top_two = shares.iloc[-2:].sum()
add_titles(
    fig,
    f"Snacks sucrés et viande-poisson-œufs : {fmt_number(top_two)} % du périmètre à eux deux",
    f"Part de chaque rayon parmi les {fmt_number(len(df))} produits du périmètre.",
)
fig.tight_layout(rect=(0, 0.04, 1, 0.89))
fig.savefig(SAVING_PATH / "g1_poids_des_rayons.png")
plt.show()

**Ce qu'on voit** : deux rayons, les snacks sucrés (21,6 %) et la viande-poisson-œufs (17,6 %), représentent 39 % du périmètre ; le plus petit, les snacks salés, compte encore 27 774 produits.

**Ce que ça implique pour NutriScope** : une moyenne calculée sur tout le périmètre serait tirée par les snacks sucrés. Il faut donc toujours raisonner **par rayon**, ce que fait le moteur de substitution.

**Limite** : c'est le poids des rayons **dans Open Food Facts** (ce que les contributeurs ont saisi), pas dans les ventes ni dans les caddies : un produit acheté chaque jour compte autant qu'un produit rare.

In [ ]:
PANELS = [
    ("energy-kcal_100g", "Énergie (kcal / 100 g)"),
    ("sugars_100g", "Sucres (g / 100 g)"),
    ("saturated-fat_100g", "Graisses saturées (g / 100 g)"),
    ("salt_100g", "Sel (g / 100 g)"),
]

# Même ordre de rayons sur les 4 graphiques : du moins sucré (en bas) au plus sucré (en haut)
order = df.groupby("rayon")["sugars_100g"].median().sort_values().index.tolist()


def draw_box(ax, col, label):
    """Dessine une boîte à moustaches par rayon pour la colonne `col`."""
    data = [df.loc[df["rayon"] == rayon, col].dropna() for rayon in order]
    ax.boxplot(
        data,
        orientation="horizontal",
        showfliers=False,
        widths=0.55,
        patch_artist=True,
        boxprops=dict(facecolor=BLUE_LIGHT, edgecolor=BLUE, linewidth=1.2),
        whiskerprops=dict(color=BLUE, linewidth=1.2),
        capprops=dict(color=BLUE, linewidth=1.2),
        medianprops=dict(color=BLUE_DARK, linewidth=2.2),
    )
    ax.set_yticks(range(1, len(order) + 1), order)
    ax.set_title(label, loc="left", fontsize=10.5, color=INK_SECONDARY)
    ax.grid(axis="y", visible=False)
    ax.set_xlim(left=0)


fig, axes = plt.subplots(2, 2, figsize=(12, 8), sharey=True)
for ax, (col, label) in zip(axes.flat, PANELS):
    draw_box(ax, col, label)

add_titles(
    fig,
    "Le sucre se concentre dans les snacks sucrés, le sel dans la viande et les snacks salés",
    f"Répartition par rayon des {fmt_number(len(df))} produits. Boîte : 50 % central des produits ; "
    "trait foncé : médiane. Valeurs extrêmes masquées (voir partie 4).",
)
fig.tight_layout(rect=(0, 0.04, 1, 0.89))
fig.savefig(SAVING_PATH / "g2_nutriments_par_rayon.png", dpi=150)
plt.show()

**Ce qu'on voit** : les snacks sucrés sont à part pour le sucre (médiane 37 g / 100 g, contre 7 g pour les boissons et moins de 6 g ailleurs) ; le sel culmine dans la viande-poisson-œufs (médiane 1,5 g) et les snacks salés (1,3 g) ; les produits laitiers dépassent les snacks sucrés en graisses saturées (11 g contre 6,4 g).

**Ce que ça implique pour NutriScope** : chaque rayon a son nutriment « à surveiller ». Une substitution doit donc comparer **le nutriment qui pèse dans ce rayon** (le sucre pour un biscuit, le sel pour du jambon), et non toujours le même.

**Limite** : le rayon « matières grasses et sauces » est très dispersé (de la sauce légère à l'huile pure, de 0 à plus de 900 kcal) : il mélange des produits qu'on ne substitue pas l'un à l'autre, et y proposer des alternatives demandera de descendre au niveau du sous-rayon.

In [ ]:
# Nutri-Score : bleu pour les bonnes notes, gris au milieu, rouge pour les mauvaises
NUTRISCORE_COLORS = {
    "a": "#1c5cab",
    "b": "#86b6ef",
    "c": "#d9d8d2",
    "d": "#f2a3a2",
    "e": "#e34948",
}
GRADES = list(NUTRISCORE_COLORS)

grade_shares = (
    df.groupby("rayon")["nutriscore_grade"]
    .value_counts(normalize=True)
    .unstack()
    .reindex(columns=GRADES)
    .fillna(0)
    * 100
)
good_share = grade_shares["a"] + grade_shares["b"]
grade_shares = grade_shares.loc[good_share.sort_values().index]

fig, ax = plt.subplots(figsize=(11, 6))
left = pd.Series(0.0, index=grade_shares.index)
for grade in GRADES:
    values = grade_shares[grade]
    ax.barh(
        grade_shares.index, values, left=left, height=0.7,
        color=NUTRISCORE_COLORS[grade], edgecolor=SURFACE, linewidth=2, label=grade.upper(),
    )
    for rayon, value in values.items():
        if value >= 6:   # pas d'étiquette dans les segments trop étroits
            text_color = "white" if grade in ("a", "e") else INK
            ax.text(left[rayon] + value / 2, rayon, f"{fmt_number(value)} %",
                    ha="center", va="center", fontsize=8.5, color=text_color)
    left += values

ax.set_xlim(0, 100)
ax.xaxis.set_visible(False)
ax.grid(False)
ax.spines["bottom"].set_visible(False)
ax.tick_params(axis="y", length=0)
ax.legend(ncols=5, loc="lower left", bbox_to_anchor=(0, 1.0), frameon=False, handlelength=1.2)

graded = df.groupby("rayon")["nutriscore_grade"].apply(lambda s: s.notna().mean() * 100)
add_titles(
    fig,
    f"Trouver un A ou un B : {fmt_number(good_share['Fruits et légumes'])} % des fruits et légumes, "
    f"mais {fmt_number(good_share['Snacks sucrés'])} % des snacks sucrés",
    f"Répartition des Nutri-Scores parmi les produits notés de chaque rayon "
    f"({fmt_number(graded.min())} % à {fmt_number(graded.max())} % des produits selon le rayon). "
    "Rayons triés par part de A + B.",
)
fig.tight_layout(rect=(0, 0.04, 1, 0.86))
fig.savefig(SAVING_PATH / "g3_nutriscore_par_rayon.png", dpi=150)
plt.show()

**Ce qu'on voit** : 60 % des fruits et légumes et 50 % des céréales et pommes de terre sont notés A ou B ; à l'opposé, seuls 2 % des snacks sucrés le sont, et 93 % y sont notés D ou E.

**Ce que ça implique pour NutriScope** : dans les snacks sucrés, proposer « une alternative A ou B » échouera presque toujours. Le moteur de substitution doit viser le **mieux relatif** dans le rayon (passer d'un E à un D, ou au D le moins sucré), sinon il ne proposera rien au persona Philipe au rayon biscuits.

**Limite** : les parts sont calculées sur les produits **notés** ; dans les boissons, 21 % des produits n'ont pas de Nutri-Score, et ceux-là ne sont pas représentés.

## Partie 2 — Où les données sont-elles fiables ?

In [ ]:
# Noms des nutriments à l'affichage (utilisés dans les parties 2 et 3)
NUTRIENT_LABELS = {
    "energy-kcal_100g": "Énergie",
    "fat_100g": "Graisses",
    "saturated-fat_100g": "Graisses sat.",
    "carbohydrates_100g": "Glucides",
    "sugars_100g": "Sucres",
    "fiber_100g": "Fibres",
    "proteins_100g": "Protéines",
    "salt_100g": "Sel",
}

In [ ]:
from matplotlib.colors import LinearSegmentedColormap

# Une seule teinte, du très clair (peu de manquants) au foncé (beaucoup de manquants)
BLUE_SCALE = LinearSegmentedColormap.from_list(
    "blue_scale", ["#fcfcfb", "#cde2fb", "#86b6ef", "#3987e5", "#1c5cab", "#0d366b"]
)

missing = df.groupby("rayon")[list(NUTRIENT_LABELS)].apply(lambda x: x.isna().mean() * 100)
missing = missing.sort_values("fiber_100g")
missing.columns = [NUTRIENT_LABELS[c] for c in missing.columns]

fig, ax = plt.subplots(figsize=(11, 6))
ax.imshow(missing.values, cmap=BLUE_SCALE, vmin=0, vmax=100, aspect="auto")

for i, rayon in enumerate(missing.index):
    for j, nutrient in enumerate(missing.columns):
        value = missing.loc[rayon, nutrient]
        text_color = "white" if value > 50 else INK
        ax.text(j, i, f"{fmt_number(value, 1)} %", ha="center", va="center", fontsize=8.5, color=text_color)

ax.set_xticks(range(len(missing.columns)), missing.columns)
ax.set_yticks(range(len(missing.index)), missing.index)
ax.tick_params(length=0)
ax.grid(False)
for spine in ax.spines.values():
    spine.set_visible(False)

others_max = missing.drop(columns="Fibres").max().max()
add_titles(
    fig,
    f"Les fibres manquent dans {fmt_number(missing['Fibres'].min(), 1)} % à "
    f"{fmt_number(missing['Fibres'].max(), 1)} % des produits selon le rayon, "
    f"les autres nutriments dans {fmt_number(others_max, 1)} % au plus",
    "Part des produits sans valeur, par rayon et par nutriment. Plus la case est foncée, plus la donnée manque.",
)
fig.tight_layout(rect=(0, 0.04, 1, 0.89))
fig.savefig(SAVING_PATH / "g4_manquants_par_rayon.png", dpi=150)
plt.show()

**Ce qu'on voit** : les fibres sont le seul vrai trou, de 29,5 % de manquants (céréales et pommes de terre) à 62,3 % (viande-poisson-œufs) ; tous les autres nutriments manquent dans 7,0 % des produits au plus (le sel des fruits et légumes).

**Ce que ça implique pour NutriScope** : les 7 nutriments obligatoires sont exploitables partout, pour le score comme pour la substitution ; les fibres ne peuvent pas servir telles quelles, ce qui confirme la stratégie du groupe (« garder NA + drapeau »).

**Limite** : « renseigné » ne veut pas dire « juste » (voir partie 4) ; et en viande, une fibre absente veut sans doute dire « 0 g » plutôt que « inconnue », puisqu'il n'y en a pas : le manque n'a pas le même sens d'un rayon à l'autre.

In [ ]:
top_brands = df["brand"].value_counts().head(15).index
in_top = df["brand"].isin(top_brands)

fiber_rate = (
    df[in_top].groupby("brand")["fiber_100g"]
    .apply(lambda s: s.notna().mean() * 100)
    .sort_values()
)
fiber_rate.index = fiber_rate.index.str.title()

# Deux repères : les autres marques, et les produits sans marque
references = pd.Series({
    "Sans marque": df.loc[df["brand"].isna(), "fiber_100g"].notna().mean() * 100,
    "Autres marques": df.loc[df["brand"].notna() & ~in_top, "fiber_100g"].notna().mean() * 100,
})

# Les 7 nutriments obligatoires, pour le sous-titre
mandatory = [c for c in NUTRIENT_LABELS if c != "fiber_100g"]
mandatory_rate = df[in_top].groupby("brand")[mandatory].apply(lambda x: x.notna().all(axis=1).mean() * 100)

labels = list(references.index) + list(fiber_rate.index)
values = list(references.values) + list(fiber_rate.values)
colors = [MUTED] * len(references) + [BLUE] * len(fiber_rate)
positions = list(range(len(references))) + [p + len(references) + 0.6 for p in range(len(fiber_rate))]

fig, ax = plt.subplots(figsize=(12, 7))
ax.barh(positions, values, color=colors, height=0.7)
for position, value in zip(positions, values):
    ax.text(value + 1, position, f"{fmt_number(value)} %", va="center", fontsize=9.5, color=INK_SECONDARY)

ax.set_yticks(positions, labels)
ax.set_xlim(0, 110)
ax.xaxis.set_visible(False)
ax.grid(False)
ax.spines["bottom"].set_visible(False)
ax.tick_params(axis="y", length=0)

add_titles(
    fig,
    f"Fibres renseignées : {fmt_number(references['Sans marque'])} % des produits sans marque, "
    f"de {fmt_number(fiber_rate.min())} à {fmt_number(fiber_rate.max())} % chez les 15 premières marques",
    f"Part des produits dont les fibres sont renseignées. Pour ces 15 marques, les 7 nutriments obligatoires "
    f"sont, eux, complets dans au moins {fmt_number(mandatory_rate.min(), 1)} % des produits.",
)
fig.tight_layout(rect=(0, 0.04, 1, 0.89))
fig.savefig(SAVING_PATH / "g5_fibres_par_marque.png", dpi=150)
plt.show()

**Ce qu'on voit** : chez les 15 premières marques, les 7 nutriments obligatoires sont complets dans au moins 95,8 % des produits ; les fibres font la différence, de 40 % (Le Gaulois) à 95 % (Auchan). Les produits sans marque tombent à 20 %, les autres marques sont à 57 %.

**Ce que ça implique pour NutriScope** : la fiabilité d'une fiche dépend de son producteur ; les 97 267 produits sans marque (21 % du périmètre) sont les moins renseignés, et l'application doit le signaler à l'utilisateur, comme le demandait le client (« afficher les sources et données manquantes »).

**Limite** : la marque retenue est la **première** marque nettoyée (règle `get_first_brand`), et les variantes d'écriture d'une même marque (« m & s », « marks & spencer »…) restent comptées séparément.

## Partie 3 — Quels nutriments vont avec le Nutri-Score ?

La lettre est traduite en nombre (A = 1 … E = 5) : plus le nombre est grand, plus la note est mauvaise.
On mesure le lien avec une **corrélation de rang** (Spearman), entre −1 et +1 : elle répond à « quand le nutriment augmente, la note se dégrade-t-elle ? », sans supposer que le lien est une ligne droite.

In [ ]:
from matplotlib.colors import LinearSegmentedColormap

# Deux teintes opposées et un gris neutre au milieu (0 = aucun lien)
DIVERGING = LinearSegmentedColormap.from_list(
    "diverging", ["#1c5cab", "#86b6ef", "#f0efec", "#f2a3a2", "#e34948"]
)

grade_number = df["nutriscore_grade"].map({"a": 1, "b": 2, "c": 3, "d": 4, "e": 5})


def spearman(x, y):
    """Corrélation de rang : la corrélation classique calculée sur les rangs, sans les vides."""
    both = x.notna() & y.notna()
    return x[both].rank().corr(y[both].rank())


rows = {"Tous rayons": {col: spearman(df[col], grade_number) for col in NUTRIENT_LABELS}}
for rayon, group in df.groupby("rayon"):
    rows[rayon] = {col: spearman(group[col], grade_number[group.index]) for col in NUTRIENT_LABELS}

correlations = pd.DataFrame(rows).T
correlations.columns = [NUTRIENT_LABELS[c] for c in correlations.columns]

fig, ax = plt.subplots(figsize=(11, 6.5))
ax.imshow(correlations.values, cmap=DIVERGING, vmin=-1, vmax=1, aspect="auto")

for i, rayon in enumerate(correlations.index):
    for j, nutrient in enumerate(correlations.columns):
        value = correlations.loc[rayon, nutrient]
        text_color = "white" if abs(value) > 0.6 else INK
        weight = "bold" if rayon == "Tous rayons" else "normal"
        ax.text(j, i, fmt_number(value, 2), ha="center", va="center", fontsize=8.5,
                color=text_color, fontweight=weight)

ax.axhline(0.5, color=SURFACE, linewidth=4)   # sépare la ligne "Tous rayons" des rayons
ax.set_xticks(range(len(correlations.columns)), correlations.columns)
ax.set_yticks(range(len(correlations.index)), correlations.index)
ax.tick_params(length=0)
ax.grid(False)
for spine in ax.spines.values():
    spine.set_visible(False)

overall_max = correlations.loc["Tous rayons"].abs().max()
within_max = correlations.drop(index="Tous rayons").abs().max().max()
add_titles(
    fig,
    f"Pris sur tout le périmètre, aucun nutriment ne dépasse {fmt_number(overall_max, 2)} ; "
    f"au sein d'un rayon, le lien monte jusqu'à {fmt_number(within_max, 2)}",
    "Corrélation de rang entre chaque nutriment et le Nutri-Score (A = 1 … E = 5). "
    "Rouge : plus il y en a, plus la note se dégrade. Bleu : plus il y en a, meilleure est la note.",
)
fig.tight_layout(rect=(0, 0.04, 1, 0.89))
fig.savefig(SAVING_PATH / "g6_correlations_nutriscore.png", dpi=150)
plt.show()

**Ce qu'on voit** : sur tout le périmètre, l'énergie et les graisses saturées sont les plus liées à une mauvaise note (0,48), les fibres vont avec une meilleure note (−0,11) et les protéines ne disent rien (0,00). Au sein des rayons, un nutriment domine souvent : le sel dans la viande-poisson-œufs (0,80), le sucre dans les boissons (0,75).

**Ce que ça implique pour NutriScope** : pour le modèle du TP14, aucun nutriment ne suffit seul, et **le rayon doit être une variable du modèle** (ou un modèle par rayon) : le même nutriment ne pèse pas pareil d'un rayon à l'autre.

**Limite** : le Nutri-Score est **calculé** à partir de ces nutriments, ces liens sont donc attendus et ne « découvrent » rien ; certaines valeurs surprennent (dans les matières grasses, plus d'énergie va avec une note un peu meilleure, sans doute les huiles végétales, mieux notées que le beurre) et demandent à être vérifiées avant d'en conclure quoi que ce soit.

## Partie 4 — Valeurs extrêmes : vraies ou erreurs ?

Une valeur **extrême** est rare mais possible (un miel à 80 g de sucre pour 100 g) : on la garde.
Une valeur **incohérente** est impossible quel que soit le produit : c'est forcément une erreur de saisie qui a échappé au nettoyage.

In [ ]:
# Trois incohérences impossibles quel que soit le produit
sugars_above_carbs = df["sugars_100g"] > df["carbohydrates_100g"] + 0.5   # les sucres font partie des glucides
macro_sum = df[["fat_100g", "carbohydrates_100g", "proteins_100g", "salt_100g"]].sum(axis=1)   # les valeurs connues suffisent : elles sont toutes >= 0
total_above_100 = macro_sum > 105                                           # plus de 100 g dans 100 g (5 g de tolérance)
zero_kcal_with_fat = (df["energy-kcal_100g"] == 0) & (df["fat_100g"] >= 5)  # 5 g de graisses = au moins 45 kcal

checks = pd.Series({
    "Sucres supérieurs aux glucides": sugars_above_carbs.sum(),
    "Graisses + glucides + protéines + sel > 105 g": total_above_100.sum(),
    "0 kcal avec au moins 5 g de graisses": zero_kcal_with_fat.sum(),
})
any_issue = sugars_above_carbs | total_above_100 | zero_kcal_with_fat
checks = checks.sort_values()

fig, ax = plt.subplots(figsize=(10, 5.5))
positions = [0] + [p + 1.6 for p in range(len(checks))]
labels = ["Au moins une incohérence"] + list(checks.index)
values = [any_issue.sum()] + list(checks.values)
colors = [BLUE_DARK] + [BLUE] * len(checks)

ax.barh(positions, values, color=colors, height=0.7)
for position, value in zip(positions, values):
    ax.text(value + 8, position, fmt_number(value), va="center", fontsize=9.5, color=INK_SECONDARY)

ax.set_yticks(positions, labels)
ax.set_xlim(0, max(values) * 1.15)
ax.xaxis.set_visible(False)
ax.grid(False)
ax.spines["bottom"].set_visible(False)
ax.tick_params(axis="y", length=0)

add_titles(
    fig,
    f"{fmt_number(any_issue.sum())} produits incohérents ont passé le nettoyage : "
    "il faut un drapeau de fiabilité",
    f"Nombre de produits par type d'incohérence ({fmt_number(any_issue.mean() * 100, 2)} % du périmètre). "
    "Un produit peut cumuler plusieurs incohérences : le total les compte une seule fois.",
)
fig.tight_layout(rect=(0, 0.05, 1, 0.86))
fig.savefig(SAVING_PATH / "g7_incoherences.png", dpi=150)
plt.show()

**Ce qu'on voit** : 846 produits (0,19 % du périmètre) cumulent au moins une incohérence : 269 ont plus de sucres que de glucides, 267 dépassent 105 g de nutriments pour 100 g, 310 affichent 0 kcal avec au moins 5 g de graisses ; les snacks sucrés en comptent le plus (224).

**Ce que ça implique pour NutriScope** : le bornage du TP9 n'attrape que l'impossible **colonne par colonne** ; ces erreurs se voient en **croisant** les colonnes. Plutôt que de les supprimer, une 7e règle de `src/cleaning.py` pourrait poser un **drapeau de fiabilité** par produit, affiché dans l'application et exclu de l'entraînement du modèle.

**Limite** : les seuils (0,5 g de tolérance, 105 g, 5 g de graisses) sont des choix, à discuter ; et ces contrôles ne voient pas les erreurs plausibles (un sucre saisi à 12 g au lieu de 21 g) ni les vraies valeurs extrêmes, comme les 3 051 miels à près de 80 g de sucre, qui sont justes.

## Synthèse — les 7 messages de l'EDA

1. **Poids des rayons** : snacks sucrés et viande-poisson-œufs font 39 % du périmètre : on raisonne toujours par rayon.
2. **Contenu des rayons** : chaque rayon a son nutriment à surveiller (le sucre pour les snacks sucrés, le sel pour la viande, les graisses saturées pour les produits laitiers).
3. **Nutri-Score** : 60 % de A ou B dans les fruits et légumes, 2 % dans les snacks sucrés : la substitution doit viser le « mieux relatif ».
4. **Manquants** : seules les fibres manquent vraiment (29,5 % à 62,3 % selon le rayon) ; les 7 nutriments obligatoires sont fiables.
5. **Marques** : sans marque, les fibres ne sont renseignées que dans 20 % des cas : la fiabilité dépend du producteur.
6. **Corrélations** : aucun nutriment ne suffit seul (0,48 au plus), mais le lien monte à 0,80 au sein d'un rayon : le rayon est une variable clé du futur modèle.
7. **Incohérences** : 846 produits impossibles ont passé le nettoyage : un drapeau de fiabilité est nécessaire.